In [1]:
import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt
%matplotlib inline
import cv2


In [2]:
cap = cv2.VideoCapture('video.mp4') 

arr = np.empty((0, 1944), int)   # Initializing 1944 dimensional array to store 'flattened' color histograms
D=dict()   # To store the original frame (array)
count=0    # Counting the number of frames
start_time = time.time()
while cap.isOpened():
    
    # Read the video file.
    ret, frame = cap.read()
    
    # If we got frames.
    if ret == True:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)  # Rearraning to get frames in RGB order
        D[count] = frame_rgb   # Storing each frame (array) to D , so that we can identify key frames later 
        
        # Dividing a frame into 3*3 i.e 9 blocks
        height, width, channels = frame_rgb.shape

        if height % 3 == 0:
            h_chunk = int(height/3)
        else:
            h_chunk = int(height/3) + 1

        if width % 3 == 0:
            w_chunk = int(width/3)
        else:
            w_chunk = int(width/3) + 1

        h=0
        w= 0 
        feature_vector = []
        for a in range(1,4):
            h_window = h_chunk*a
            for b in range(1,4):
                frame = frame_rgb[h : h_window, w : w_chunk*b , :]
                hist = cv2.calcHist(frame, [0, 1, 2], None, [6, 6, 6], [0, 256, 0, 256, 0, 256]) # Finding histograms for each block  
                hist1= hist.flatten()  # Flatten the hist to one-dimensinal vector 
                feature_vector += list(hist1)
                w = w_chunk*b
                
            h = h_chunk*a
            w= 0

                
        arr =np.vstack((arr, feature_vector )) # Appending each one-dimensinal vector to generate N*M matrix (where N is number of frames
          #and M is 1944) 
        count+=1
    else:
        break

print("--- %s seconds ---" % (time.time() - start_time))

final_arr = arr.transpose() # Transposing so that i will have all frames in columns i.e M*N dimensional matrix 
#where M is 1944 and N is number of frames
print(final_arr.shape)
print(count)


--- 32.90750217437744 seconds ---
(1944, 1832)
1832


In [3]:
from scipy.sparse import csc_matrix
from scipy.sparse.linalg import svds, eigs
A = csc_matrix(final_arr, dtype=float)

# Top 63 singular values from 76082 to 508
u, s, vt = svds(A, k = 63)


In [4]:
print(u.shape, s.shape, vt.shape)

(1944, 63) (63,) (63, 1832)


In [5]:
print(list(s))

[507.5863397910355, 513.3394019469023, 542.5885461980126, 557.4615730264487, 581.1252578281193, 595.6523491133408, 625.867625112877, 667.3543038445864, 703.9369306867551, 785.0033226440836, 824.1240297172749, 830.2095332996893, 862.3243911201392, 962.1678005881331, 993.1884335353163, 1074.4879221595822, 1104.0363306894772, 1174.3795871198809, 1310.402462874377, 1429.805972926553, 1436.0497628725554, 1784.0835463882931, 1920.5743747040278, 2075.5124822673374, 2235.99300389515, 2450.850435862492, 2789.8612842475472, 3266.437872035349, 3467.4543326978155, 3703.2744228804368, 4026.512028827832, 4160.600410916417, 4339.992564704217, 4608.340768433541, 4872.711772927266, 5124.862670258989, 5276.309542523111, 5668.146768603118, 5931.5079220896405, 5952.291087578095, 6235.342567902835, 6697.298321083398, 6776.947714481711, 7065.201258998799, 7853.1214805878935, 8067.823543248026, 8691.548708604727, 9140.661744359932, 9938.174572146698, 10316.757223295499, 10792.030201567131, 11187.863261213432

In [6]:
v1_t = vt.transpose()

projections = v1_t @ np.diag(s) # Projection along orthonormal basis
print(projections.shape)

(1832, 63)


In [7]:
# Dynamic Clustering 
f=projections
C = dict() # To store frames in respective cluster
for i in range(f.shape[0]):
    C[i] = np.empty((0,63), int)
    
# Initializaton    
C[0] = np.vstack((C[0], f[0]))   
C[0] = np.vstack((C[0], f[1]))

E = dict() # To store centroids of each cluster
for i in range(projections.shape[0]):
    E[i] = np.empty((0,63), int)
    
E[0] = np.mean(C[0], axis=0) # Finding centroid of C[0] cluster

count = 0
for i in range(2,f.shape[0]):
    similarity = np.dot(f[i], E[count])/( (np.dot(f[i],f[i]) **.5) * (np.dot(E[count], E[count]) ** .5)) # Cosine Similarity
    
     
    
    if similarity < 0.9: # Threshhold Value

        # IF NOT SIMILAR, assign current data point to a new cluster          
        count+=1         
        C[count] = np.vstack((C[count], f[i])) 
        E[count] = np.mean(C[count], axis=0)   
    else:  # IF SIMILAR, assign current data point to current cluster formed 
        C[count] = np.vstack((C[count], f[i])) 
        E[count] = np.mean(C[count], axis=0)          

In [8]:
b = []  # Finding number of data points in each cluster formed.

# Extracting Key Frames from dense clusters
for i in range(f.shape[0]):
    b.append(C[i].shape[0])

last = b.index(0)  #  0 in b indicates that all required clusters have been formed , so we can delete these from C
b1=b[:last ] # The size of each cluster.

In [9]:
res = [idx for idx, val in enumerate(b1) if val >= 25] # Choosing Dense cluster with atleast 25 frames
print(len(res)) # 

25


In [10]:
GG = C # Copying the elements of C to GG, 

# Labelling Each Cluster
for i in range(last):
    p1= np.repeat(i, b1[i]).reshape(b1[i],1)
    GG[i] = np.hstack((GG[i],p1))

In [11]:
# Appending each cluster to get Multidimensional array of dimension N*64, N is number of frames
F=  np.empty((0,64), int) 
for i in range(last):
    F = np.vstack((F,GG[i]))

In [12]:
#Converting F (multidimensional array)  to dataframe

colnames = []
for i in range(1, 65):
    col_name = "v" + str(i)
    colnames+= [col_name]
print(colnames)

df = pd.DataFrame(F, columns= colnames)

['v1', 'v2', 'v3', 'v4', 'v5', 'v6', 'v7', 'v8', 'v9', 'v10', 'v11', 'v12', 'v13', 'v14', 'v15', 'v16', 'v17', 'v18', 'v19', 'v20', 'v21', 'v22', 'v23', 'v24', 'v25', 'v26', 'v27', 'v28', 'v29', 'v30', 'v31', 'v32', 'v33', 'v34', 'v35', 'v36', 'v37', 'v38', 'v39', 'v40', 'v41', 'v42', 'v43', 'v44', 'v45', 'v46', 'v47', 'v48', 'v49', 'v50', 'v51', 'v52', 'v53', 'v54', 'v55', 'v56', 'v57', 'v58', 'v59', 'v60', 'v61', 'v62', 'v63', 'v64']


In [13]:
df['v64']= df['v64'].astype(int)  # Converting the cluster level from float type to integer type

In [14]:
df1 =  df[df.v64.isin(res)]   # Filtering the frames from the clusters that have more than 25 frames in it

In [15]:
new = df1.groupby('v64').tail(1)['v64'] # Taking the last shot of each frame to identify the key-frame

In [16]:

new1 = new.index # Finding key-frames                                    

In [17]:
# Output the frames in png format

for c in new1:
    frame_rgb1 = cv2.cvtColor(D[c], cv2.COLOR_RGB2BGR) #since cv consider image in BGR order
    frame_num_chr = str(c)
    file_name = 'frame'+ frame_num_chr +'.png'
    cv2.imwrite(file_name, frame_rgb1)
    